# 베이스라인


데이터와 모델 모두 최소한으로 가공해서 학습한다. 

예측을 내고 캐글에 제출까지 해본다.

## 데이터 타입 범위 표

<img src = https://user-images.githubusercontent.com/89520805/166191804-c083cbd2-5f3f-4850-8f4f-41fc8982a090.png height = 500 width = 550>

# 1. 데이터 불러오기

In [8]:
import numpy as np
import pandas as pd

path = '/content/drive/MyDrive/data/instacart-market-basket-analysis/' # csv파일이 들어있는 폴더 경로 지정

# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# 데이터 타입의 범위를 파악해서 효율적인 데이터 타입으로 변환하는 함수
def reduce_memory(df):
    start_mem_usg = df.memory_usage().sum() / 1024**2 
    print("Memory usage of properties dataframe is :",start_mem_usg," MB")
    
    for col in df.columns:
        if df[col].dtypes in ["int64", "int32", "int16"]:
            
            cmin = df[col].min()
            cmax = df[col].max()
            
            if cmin > np.iinfo(np.int8).min and cmax < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            
            elif cmin > np.iinfo(np.int16).min and cmax < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            
            elif cmin > np.iinfo(np.int32).min and cmax < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        
        if df[col].dtypes in ["float64", "float32"]:
            
            cmin = df[col].min()
            cmax = df[col].max()
            
            if cmin > np.finfo(np.float16).min and cmax < np.finfo(np.float16).max:
                df[col] = df[col].astype(np.float16)
            
            elif cmin > np.finfo(np.float32).min and cmax < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
    
    print("")
    print("___MEMORY USAGE AFTER COMPLETION:___")
    mem_usg = df.memory_usage().sum() / 1024**2 
    print("Memory usage is: ",mem_usg," MB")
    print("This is ",100*mem_usg/start_mem_usg,"% of the initial size")
    
    return df

데이터 타입에 따라서 메모리를 차지하는 용량이 차이가 난다.

판다스 데이터 프레임은 기본적으로 가장 큰 범위인 int64를 사용한다.
따라서 데이터에 따라서 더 작은 범위의 데이터타입을 사용하면 메모리를 훨씬 효율적으로
사용할 수 있다. 

In [6]:
# 데이터를 불러오면서 데이터타입 변환하기
priors = reduce_memory(pd.read_csv(path + 'order_products__prior.csv'))
train = reduce_memory(pd.read_csv(path + 'order_products__train.csv'))
orders = reduce_memory(pd.read_csv(path + 'orders.csv'))
products = reduce_memory(pd.read_csv(path + 'products.csv'))

Memory usage of properties dataframe is : 989.8221740722656  MB

___MEMORY USAGE AFTER COMPLETION:___
Memory usage is:  340.2514524459839  MB
This is  34.375008093235806 % of the initial size
Memory usage of properties dataframe is : 42.255279541015625  MB

___MEMORY USAGE AFTER COMPLETION:___
Memory usage is:  13.204858779907227  MB
This is  31.250198610305635 % of the initial size
Memory usage of properties dataframe is : 182.7056655883789  MB

___MEMORY USAGE AFTER COMPLETION:___
Memory usage is:  68.5147008895874  MB
This is  37.50004175784318 % of the initial size
Memory usage of properties dataframe is : 1.5164794921875  MB

___MEMORY USAGE AFTER COMPLETION:___
Memory usage is:  0.7109146118164062  MB
This is  46.879276342268376 % of the initial size


# 2. 데이터 전처리 하기

데이터를 모델에 넣을 수 있게 X_train, y_train, X_test, y_test 형식으로 변환해야 한다. 

먼저 orders DF를 살펴보자. 20만 유저의 340만 건의 주문이 있다. 

유저별로 최소 4개에서 100개의 주문 내역이 있다. 모든 유저의 마지막 주문은 train과  test로 구분되어 있다. 13만 개의 주문은 train, 7.5만 개의 주문은 test이다. train데이터와 test데이터의 차이는 train데이터는 해당 주문별로 유저가 어떤 제품을 구매했는지 알 수 있다. 

따라서 우리는 train 데이터의 주문 번호와 구매한 제품을 가지고 훈련을 해서 test데이터의 주문번호 마다 어떤 제품이 들어있을지 예측해야 한다. 

제출 형식은 컬럼1이 test데이터의 order_id이고 컬럼2가 product_id를 공백으로 구분해서 넣는다. 만약 유저가 구매하지 않았다면 `None`을 넣는다. 

train 데이터는 feature와 label로 나눠야한다. 일단 필요한 컬럼으로는 

1. order_id 
2.해당 유저가 구매한 모든 제품_id 
3. 레이블 : 실제로 구매한 제품은 1, 그렇지 않으면 0. 
4. 제품이나 주문, 고객에 대한 feature들.

데이터를 가지고 훈련이 가능한 형태로 만드는 방법

- train_order
    
    orders에서 train셋인 것들만 따로 데이터프레임으로 준비.
    
    이 데이터는 주문번호, 주문자 등이 있다. 
- user_prods
    
    user_id를 인덱스로 잡고 해당 고객이 주문한 모든 제품을 set으로 담는다. 

- train_index

    train_products 즉, train셋의 제품 목록이다. order_id와 product_id를 인덱스로 준비.

아이디어는 이렇다. 최소한의 데이터는 order_id와 product_id다. 

1. X_train : 유저가 구매한 모든 제품과 마지막 order_id를 하나씩 매칭해서 데이터프레임으로 만든다. 

2. y_train : X_train을 train_products를 비교해서 맞은 건 1, 틀린 건 0으로 레이블링한다. 



1. 구하고자하는 user_id를 기준으로 해당 유저가 구매한 모든 제품을 불러온다. 

2. 데이터프레임을 생성하고 먼저 2열에 유저가 구매한 모든 제품을 넣는다. 그리고 1열에는 2열의 길이에 맞춰서 구하고자하는 order_id를 중복해서 넣는다. 

3. train_index를 가져와서 


In [41]:
# priors에 orders 추가
orders.set_index('order_id', inplace=True, drop=False)
priors = priors.join(orders, on='order_id', rsuffix='_')    # 뒤에 '_' 붙이기
priors.drop('order_id_', inplace=True, axis=1)

In [42]:
# order 분리하기
prior_orders = orders[orders['eval_set'] == 'prior'].drop('eval_set', axis=1)
train_orders = orders[orders['eval_set'] == 'train'].drop('eval_set', axis=1)
test_orders = orders[orders['eval_set'] == 'test'].drop('eval_set', axis=1)

In [14]:
train.set_index(['order_id', 'product_id'], inplace=True, drop=False)

train_orders.shape, test_orders.shape

((131209, 6), (75000, 6))

In [37]:
users = pd.DataFrame()
users['all_products'] = priors.groupby('user_id')['product_id'].unique()
users.head()

,all_products
user_id,
1,"[196, 12427, 10258, 25133, 10326, 17122, 41787..."
2,"[49451, 32792, 32139, 34688, 36735, 37646, 228..."
3,"[38596, 21903, 248, 40604, 8021, 17668, 21137,..."
4,"[22199, 25146, 1200, 17769, 43704, 37646, 1186..."
5,"[27344, 24535, 43693, 40706, 16168, 21413, 139..."


In [66]:
def features(selected_orders, labels_given=False):
    order_list = []
    product_list = []
    labels = []

    train_index = set(train.index) # order_id마다 구매한 모든 제품과 매칭되어 반환
    """
        (1, 10246),
        (1, 11109),
        (1, 13176)
    """
    i=0
    for row in selected_orders.to_numpy():    # orders를 한 행씩 꺼내온다.
        i+=1
        if i%10000 == 0: print('order row',i)
        order_id = row[0].astype(np.int32)
        # print('order_id: ',order_id)
        user_id = row[1].astype(np.int32)
        # print('user_id: ',user_id)
        user_prods = users['all_products'][user_id] # 해당 유저가 구매한 모든 제품 
        # print('user_prods: ',user_prods)
        product_list.extend(user_prods)
        # print('product_list: ',product_list)
        order_list += [order_id] * len(user_prods) # 유저가 구매한 모든 제품의 개수만큼 order_id를 복사한다.  
        # print('order_list: ',order_list)
        if labels_given:
            labels += [(order_id, prod) in train_index for prod in user_prods]
        
    df = pd.DataFrame({'order_id':order_list, 'product_id':product_list}, dtype=np.int32)
    labels = np.array(labels, dtype=np.int8)
    del order_list
    del product_list
    return (df, labels)

In [67]:
df_train, labels = features(train_orders, labels_given=True)

order row 10000
order row 20000
order row 30000
order row 40000
order row 50000
order row 60000
order row 70000
order row 80000
order row 90000
order row 100000
order row 110000
order row 120000
order row 130000


In [78]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df_train, labels)

In [79]:
X_test

,order_id,product_id
28647,1368829,22312
64960,2721390,23622
621736,2174636,335
2817297,2534439,43789
42580,595633,18656
...,...,...
1916115,1892044,14704
6913944,719106,49510
6383442,3150761,24544
5955700,202277,36426


In [70]:
df_train.shape, labels.shape

((8474661, 2), (8474661,))

In [ ]:
df_train, labels = features(train_orders, labels_given=True)
df_test, _ = features(test_orders)

# 모델 학습, 평가

In [71]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from xgboost import XGBClassifier, plot_importance

import lightgbm as lgbm
from lightgbm import LGBMClassifier, plot_importance



In [ ]:
xgbc = XGBClassifier()
xgbc.fit(X_train, y_train)

# preds = xgbc.predict(X_test)
# preds_proba = xgbc.predict_proba(X_test)[:, 1]

In [81]:
lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)

LGBMClassifier()

In [83]:
preds = lgbm.predict(df_test)

df_test['pred'] = preds

In [84]:
df_test.head()

,order_id,product_id,pred
0,2774568,38596,0
1,2774568,21903,0
2,2774568,248,0
3,2774568,40604,0
4,2774568,8021,0


In [87]:
%%time 
TRESHOLD = 0.22

d = dict()
for row in df_test.itertuples():
    if row.pred > TRESHOLD:
        try:
            d[row.order_id] += ' ' + str(row.product_id)
        except:
            d[row.order_id] = str(row.product_id)

for order in test_orders.order_id:
    if order not in d:
        d[order] = 'None'

sub = pd.DataFrame.from_dict(d, orient='index')

sub.reset_index(inplace=True)
sub.columns = ['order_id', 'products']
sub.to_csv('sub.csv', index=False)

CPU times: user 4.84 s, sys: 16.3 ms, total: 4.85 s
Wall time: 4.93 s


In [90]:
# 캐글 API 사용하기
!mkdir -p ~/.kaggle
%cd /content/drive/MyDrive/
!chmod 600 kaggle.json
!cp kaggle.json ~/.kaggle
%cd /content

/content/drive/MyDrive
/content


In [91]:
# submit코드
!kaggle competitions submit -c instacart-market-basket-analysis -f sub.csv -m "2"

100% 928k/928k [00:06<00:00, 153kB/s]
Successfully submitted to Instacart Market Basket Analysis